# ⚾ MLB 2024 Backtest — NRFI/YRFI + Starter K Total Models

**Data:** Baseball Savant Statcast (pitch-by-pitch) + FanGraphs via `pybaseball`  
**Models:** Logistic Regression (NRFI/YRFI) · Ridge Regression (K Total O/U)  
**Split:** Train on first 60% of season → Test on final 40%  
**Runtime:** ~20 min first run (downloads 2024 Statcast season) · ~2 min cached  

---
### Features Used
| NRFI/YRFI | K Total |
|---|---|
| Starter 1st-inning K%, BB%, wOBA allowed | Pitcher avg Ks/game |
| Opposing lineup wOBA + K% (1st inning) | Pitcher K% (full season) |
| Park factor | Opponent team K% |
| Weather: wind speed × direction + temp | Park factor |

## 1 · Install Dependencies

In [ ]:
%%capture
!pip install pybaseball pandas numpy scikit-learn matplotlib seaborn

## 2 · Imports & Constants

In [ ]:
import os, time, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.dates as mdates
import seaborn as sns
import pybaseball as pb
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, mean_absolute_error

pb.cache.enable()
warnings.filterwarnings('ignore')
os.makedirs('cache', exist_ok=True)

MONTHLY_RANGES = [
    ('2024-03-20', '2024-04-30'), ('2024-05-01', '2024-05-31'),
    ('2024-06-01', '2024-06-30'), ('2024-07-01', '2024-07-31'),
    ('2024-08-01', '2024-08-31'), ('2024-09-01', '2024-10-01'),
]

PARK_FACTORS = {
    'COL':115,'CIN':110,'BOS':108,'PHI':107,'TEX':106,'BAL':104,'NYY':103,
    'CHC':103,'MIL':102,'TOR':101,'ATL':100,'HOU':100,'LAD':100,'WSH':100,
    'CLE':100,'DET':99,'MIN':99,'STL':99,'ARI':99,'MIA':98,'NYM':98,
    'OAK':98,'PIT':98,'SEA':97,'TB':97,'CWS':96,'KC':96,'LAA':96,'SF':95,'SD':94,
}

WIND_DIR_MAP = {
    'Out to CF':1.0,'Out to RF':0.8,'Out to LF':0.8,
    'In from CF':-1.0,'In from LF':-0.8,'In from RF':-0.8,
    'L to R':0.1,'R to L':0.1,'Calm':0.0,'':0.0,
}

NRFI_FEATURES = [
    'avg_pitcher_k_pct','avg_pitcher_bb_pct','avg_pitcher_dom',
    'avg_opp_woba_1st','avg_opp_k_pct_1st','avg_opp_woba_full',
    'park_factor','weather_offense_factor',
    'hp_woba_allowed_1st','ap_woba_allowed_1st',
    'hp_k_pct_1st','ap_k_pct_1st',
]

K_FEATURES = [
    'avg_k_per_game','k_pct_full','k_consistency',
    'opp_k_pct','opp_woba','park_factor',
]

print('✅  Setup complete')

## 3 · Load Data
Downloads 2024 Statcast season from Baseball Savant (~20 min first run, cached after).

In [ ]:
def load_statcast():
    cache = 'cache/statcast_2024.csv'
    if os.path.exists(cache):
        print('📂  Loading from cache...')
        df = pd.read_csv(cache, low_memory=False)
        df['game_date'] = pd.to_datetime(df['game_date'])
        return df
    print('⬇️   Downloading 2024 Statcast (one-time ~20 min)...')
    chunks = []
    for i,(s,e) in enumerate(MONTHLY_RANGES,1):
        print(f'  [{i}/6] {s} → {e}')
        try:
            chunks.append(pb.statcast(start_dt=s, end_dt=e))
            time.sleep(3)
        except Exception as ex:
            print(f'  ⚠️  {ex}')
    df = pd.concat(chunks, ignore_index=True)
    df['game_date'] = pd.to_datetime(df['game_date'])
    df.to_csv(cache, index=False)
    return df

sc = load_statcast()
print(f'\n✅  {len(sc):,} pitches | {sc["game_pk"].nunique():,} games | '
      f'{sc["game_date"].min().date()} → {sc["game_date"].max().date()}')

## 4 · Feature Engineering

In [ ]:
def build_game_log(sc):
    meta = sc.groupby('game_pk').agg(
        game_date=('game_date','first'), home_team=('home_team','first'), away_team=('away_team','first')
    ).reset_index()
    inn1 = sc[sc['inning']==1].copy()
    runs_1st = inn1.groupby('game_pk').apply(
        lambda g: (g['post_bat_score']-g['bat_score']).clip(lower=0).sum()
    ).reset_index(name='total_runs_1st')
    meta = meta.merge(runs_1st, on='game_pk', how='left')
    meta['total_runs_1st'] = meta['total_runs_1st'].fillna(0)
    meta['YRFI'] = (meta['total_runs_1st'] > 0).astype(int)
    inn1s = inn1.sort_values(['game_pk','inning_topbot','pitch_number'])
    hp = inn1s[inn1s['inning_topbot']=='Top'].groupby('game_pk')['pitcher'].first().reset_index().rename(columns={'pitcher':'home_starter_id'})
    ap = inn1s[inn1s['inning_topbot']=='Bot'].groupby('game_pk')['pitcher'].first().reset_index().rename(columns={'pitcher':'away_starter_id'})
    meta = meta.merge(hp, on='game_pk', how='left').merge(ap, on='game_pk', how='left')
    for col,default in [('wind_speed',5.0),('wind_dir','Calm'),('temperature',72.0)]:
        if col in sc.columns:
            w = sc.groupby('game_pk')[col].first().reset_index()
            meta = meta.merge(w, on='game_pk', how='left')
        if col not in meta.columns: meta[col] = default
    meta['wind_speed']  = pd.to_numeric(meta['wind_speed'],  errors='coerce').fillna(5.0)
    meta['temperature'] = pd.to_numeric(meta['temperature'], errors='coerce').fillna(72.0)
    meta['wind_dir_factor'] = meta['wind_dir'].astype(str).str.strip().map(WIND_DIR_MAP).fillna(0.0)
    meta['weather_offense_factor'] = (meta['wind_dir_factor']*(meta['wind_speed']/10.0)
                                      + (meta['temperature']-72)/30.0)
    return meta

def build_pitcher_1st_inning_stats(sc):
    pa = sc[(sc['inning']==1) & sc['events'].notna()].copy()
    stats = pa.groupby('pitcher').agg(
        pa_faced_1st=('events','count'),
        k_1st=('events', lambda x:(x=='strikeout').sum()),
        bb_1st=('events',lambda x:(x=='walk').sum()),
        woba_1st=('woba_value','mean'),
    ).reset_index()
    stats = stats[stats['pa_faced_1st']>=10].copy()
    stats['k_pct_1st']        = stats['k_1st']  / stats['pa_faced_1st']
    stats['bb_pct_1st']       = stats['bb_1st'] / stats['pa_faced_1st']
    stats['woba_allowed_1st'] = stats['woba_1st'].fillna(0.320)
    stats['pitcher_dom_1st']  = stats['k_pct_1st'] - stats['bb_pct_1st'] - stats['woba_allowed_1st']
    return stats[['pitcher','pa_faced_1st','k_pct_1st','bb_pct_1st','woba_allowed_1st','pitcher_dom_1st']]

def build_pitcher_fullgame_stats(sc):
    pa = sc[sc['events'].notna()].copy()
    game_k = pa.groupby(['game_pk','game_date','pitcher']).agg(
        k_count=('events',lambda x:(x=='strikeout').sum()),
        pa_faced=('events','count'),
    ).reset_index()
    season = game_k.groupby('pitcher').agg(
        games=('game_pk','nunique'), total_k=('k_count','sum'),
        total_pa=('pa_faced','sum'), avg_k_per_game=('k_count','mean'),
        std_k_per_game=('k_count','std'), median_k=('k_count','median'),
    ).reset_index()
    season = season[season['games']>=5].copy()
    season['k_pct_full']    = season['total_k'] / season['total_pa']
    season['k_consistency'] = 1 - (season['std_k_per_game'] / (season['avg_k_per_game']+0.01))
    return game_k, season

def build_team_batting_features(sc):
    pa = sc[sc['events'].notna()].copy()
    pa['batting_team'] = np.where(pa['inning_topbot']=='Top', pa['away_team'], pa['home_team'])
    full = pa.groupby('batting_team').agg(
        team_woba_full=('woba_value','mean'),
        team_k_pct_full=('events',lambda x:(x=='strikeout').sum()/len(x)),
        team_bb_pct_full=('events',lambda x:(x=='walk').sum()/len(x)),
    ).reset_index()
    pa1 = pa[pa['inning']==1].copy()
    pa1['batting_team'] = np.where(pa1['inning_topbot']=='Top', pa1['away_team'], pa1['home_team'])
    inn1t = pa1.groupby('batting_team').agg(
        team_woba_1st=('woba_value','mean'),
        team_k_pct_1st=('events',lambda x:(x=='strikeout').sum()/len(x)),
    ).reset_index()
    return full.merge(inn1t, on='batting_team', how='left')

def build_nrfi_dataset(game_log, pitcher_1st, team_stats):
    df = game_log.copy()
    hp = pitcher_1st.add_prefix('hp_').rename(columns={'hp_pitcher':'pitcher'})
    ap = pitcher_1st.add_prefix('ap_').rename(columns={'ap_pitcher':'pitcher'})
    df = df.merge(hp, left_on='home_starter_id', right_on='pitcher', how='left').drop(columns=['pitcher'])
    df = df.merge(ap, left_on='away_starter_id', right_on='pitcher', how='left').drop(columns=['pitcher'])
    at = team_stats.add_prefix('away_bat_').rename(columns={'away_bat_batting_team':'bt'})
    ht = team_stats.add_prefix('home_bat_').rename(columns={'home_bat_batting_team':'bt'})
    df = df.merge(at, left_on='away_team', right_on='bt', how='left').drop(columns=['bt'])
    df = df.merge(ht, left_on='home_team', right_on='bt', how='left').drop(columns=['bt'])
    df['park_factor']          = df['home_team'].map(PARK_FACTORS).fillna(100) / 100.0
    df['avg_pitcher_k_pct']    = df[['hp_k_pct_1st','ap_k_pct_1st']].mean(axis=1)
    df['avg_pitcher_bb_pct']   = df[['hp_bb_pct_1st','ap_bb_pct_1st']].mean(axis=1)
    df['avg_pitcher_dom']      = df[['hp_pitcher_dom_1st','ap_pitcher_dom_1st']].mean(axis=1)
    df['avg_opp_woba_1st']     = df[['away_bat_team_woba_1st','home_bat_team_woba_1st']].mean(axis=1)
    df['avg_opp_k_pct_1st']    = df[['away_bat_team_k_pct_1st','home_bat_team_k_pct_1st']].mean(axis=1)
    df['avg_opp_woba_full']    = df[['away_bat_team_woba_full','home_bat_team_woba_full']].mean(axis=1)
    df['game_date'] = pd.to_datetime(df['game_date'])
    return df.sort_values('game_date').reset_index(drop=True)

def build_k_dataset(game_log, game_k, season_pitcher, team_stats):
    home = game_log[['game_pk','game_date','home_team','away_team','home_starter_id']].copy()
    home['pitcher'] = home['home_starter_id']; home['batting_team'] = home['away_team']
    away = game_log[['game_pk','game_date','home_team','away_team','away_starter_id']].copy()
    away['pitcher'] = away['away_starter_id']; away['batting_team'] = away['home_team']
    starts = pd.concat([
        home[['game_pk','game_date','home_team','pitcher','batting_team']],
        away[['game_pk','game_date','home_team','pitcher','batting_team']],
    ], ignore_index=True)
    game_k['game_date'] = pd.to_datetime(game_k['game_date'])
    starts = starts.merge(game_k[['game_pk','pitcher','k_count']], on=['game_pk','pitcher'], how='inner')
    starts = starts.merge(
        season_pitcher[['pitcher','avg_k_per_game','k_pct_full','k_consistency','median_k','std_k_per_game']],
        on='pitcher', how='inner')
    opp = team_stats[['batting_team','team_k_pct_full','team_woba_full']].rename(
        columns={'team_k_pct_full':'opp_k_pct','team_woba_full':'opp_woba'})
    starts = starts.merge(opp, on='batting_team', how='left')
    starts['park_factor'] = starts['home_team'].map(PARK_FACTORS).fillna(100) / 100.0
    starts['game_date'] = pd.to_datetime(starts['game_date'])
    return starts.sort_values('game_date').reset_index(drop=True)

print('⚙️  Building features...')
game_log    = build_game_log(sc)
pitcher_1st = build_pitcher_1st_inning_stats(sc)
game_k, sp  = build_pitcher_fullgame_stats(sc)
team_stats  = build_team_batting_features(sc)
nrfi_df     = build_nrfi_dataset(game_log, pitcher_1st, team_stats)
k_df        = build_k_dataset(game_log, game_k, sp, team_stats)
nrfi_ready  = nrfi_df.dropna(subset=NRFI_FEATURES)
k_ready     = k_df.dropna(subset=K_FEATURES)
print(f'✅  NRFI rows: {len(nrfi_ready):,} | K model rows: {len(k_ready):,}')
print(f'   Full-season YRFI rate: {game_log["YRFI"].mean():.1%}')

## 5 · NRFI/YRFI Model — Backtest

In [ ]:
df = nrfi_ready.sort_values('game_date').reset_index(drop=True)
split = int(len(df)*0.60)
train, test = df.iloc[:split], df.iloc[split:]

scaler_n = StandardScaler()
X_tr = scaler_n.fit_transform(train[NRFI_FEATURES])
X_te = scaler_n.transform(test[NRFI_FEATURES])

nrfi_model = LogisticRegression(C=0.5, max_iter=2000, class_weight='balanced', random_state=42)
nrfi_model.fit(X_tr, train['YRFI'])

preds = nrfi_model.predict(X_te)
proba = nrfi_model.predict_proba(X_te)[:,1]

nrfi_results = test[['game_date','home_team','away_team','YRFI']].copy()
nrfi_results['predicted_YRFI']   = preds
nrfi_results['yrfi_probability'] = proba
nrfi_results['correct'] = (nrfi_results['predicted_YRFI']==nrfi_results['YRFI']).astype(int)

nrfi_fi = pd.DataFrame({'feature':NRFI_FEATURES,'coefficient':nrfi_model.coef_[0]})
nrfi_fi = nrfi_fi.reindex(nrfi_fi['coefficient'].abs().sort_values(ascending=False).index)

acc = accuracy_score(test['YRFI'], preds)
print(f'Train: {len(train):,} games  |  Test: {len(test):,} games')
print(f'Actual YRFI rate : {test["YRFI"].mean():.1%}')
print(f'Model accuracy   : {acc:.1%}\n')
print(classification_report(test['YRFI'], preds, target_names=['NRFI','YRFI'], digits=3))

## 6 · K Total Model — Backtest

In [ ]:
kdf = k_ready.sort_values('game_date').reset_index(drop=True)
ks  = int(len(kdf)*0.60)
ktrain, ktest = kdf.iloc[:ks], kdf.iloc[ks:]

scaler_k = StandardScaler()
Xk_tr = scaler_k.fit_transform(ktrain[K_FEATURES])
Xk_te = scaler_k.transform(ktest[K_FEATURES])

k_model = Ridge(alpha=2.0)
k_model.fit(Xk_tr, ktrain['k_count'])

predicted = k_model.predict(Xk_te)
actual    = ktest['k_count'].values
k_line    = float(ktrain['k_count'].median())

pred_over   = (predicted >= k_line).astype(int)
actual_over = (actual    >= k_line).astype(int)
ou_acc      = accuracy_score(actual_over, pred_over)

k_results = ktest[['game_date','pitcher','batting_team']].copy()
k_results['actual_k']    = actual
k_results['predicted_k'] = predicted.round(2)
k_results['error']       = (predicted - actual).round(2)
k_results['k_line']      = k_line
k_results['pred_over']   = pred_over
k_results['actual_over'] = actual_over
k_results['correct']     = (pred_over == actual_over).astype(int)

print(f'K line (train median): {k_line:.1f} Ks')
print(f'MAE                  : {mean_absolute_error(actual,predicted):.2f} Ks')
print(f'RMSE                 : {np.sqrt(np.mean((predicted-actual)**2)):.2f} Ks')
print(f'O/U accuracy         : {ou_acc:.1%}')

## 7 · Results Dashboard

In [ ]:
sns.set_theme(style='darkgrid', palette='muted', font_scale=0.95)
fig = plt.figure(figsize=(22,16))
fig.patch.set_facecolor('#0f1117')
fig.suptitle('MLB 2024 Backtest — NRFI/YRFI + Starter K Total Models',
             fontsize=18, fontweight='bold', color='white', y=0.98)
gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.48, wspace=0.38)
BLUE,PURPLE,GREEN,RED,ORANGE,BG,TEXT = '#2196F3','#9C27B0','#4CAF50','#F44336','#FF9800','#1a1d27','#e0e0e0'

def sax(ax, title):
    ax.set_facecolor(BG); ax.set_title(title, fontsize=11, fontweight='bold', color=TEXT, pad=8)
    ax.tick_params(colors=TEXT); ax.xaxis.label.set_color(TEXT); ax.yaxis.label.set_color(TEXT)
    [s.set_edgecolor('#444') for s in ax.spines.values()]

# 1. NRFI rolling accuracy
ax1 = fig.add_subplot(gs[0,:2])
nr = nrfi_results.sort_values('game_date').copy()
nr['rolling_acc'] = nr['correct'].rolling(30, min_periods=10).mean()
ov = nr['correct'].mean()
ax1.fill_between(nr['game_date'], nr['rolling_acc'], alpha=0.2, color=BLUE)
ax1.plot(nr['game_date'], nr['rolling_acc'], color=BLUE, linewidth=2, label='30-game rolling')
ax1.axhline(ov, color=RED, linestyle='--', linewidth=1.5, label=f'Overall: {ov:.1%}')
ax1.axhline(0.50, color='gray', linestyle=':', alpha=0.6, label='50% baseline')
ax1.set_ylim(0.35,0.85); ax1.set_ylabel('Accuracy', color=TEXT)
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
ax1.legend(facecolor=BG, labelcolor=TEXT, fontsize=9)
sax(ax1, 'NRFI/YRFI — 30-Game Rolling Accuracy')

# 2. NRFI feature importance
ax2 = fig.add_subplot(gs[0,2])
fi = nrfi_fi.head(8).iloc[::-1]
ax2.barh(fi['feature'], fi['coefficient'], color=[GREEN if c>0 else RED for c in fi['coefficient']], height=0.6)
ax2.axvline(0, color='white', linewidth=0.8); ax2.set_xlabel('Coefficient', color=TEXT)
sax(ax2, 'NRFI Feature Importance\n(+→YRFI  −→NRFI)')

# 3. Confusion matrix
ax3 = fig.add_subplot(gs[1,0])
cm = confusion_matrix(nrfi_results['YRFI'], nrfi_results['predicted_YRFI'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax3,
            xticklabels=['Pred NRFI','Pred YRFI'], yticklabels=['Act NRFI','Act YRFI'],
            linewidths=0.5, linecolor='#333', annot_kws={'size':13,'weight':'bold'})
ax3.set_facecolor(BG); ax3.tick_params(colors=TEXT)
ax3.set_title('NRFI Confusion Matrix', fontsize=11, fontweight='bold', color=TEXT, pad=8)

# 4. Probability separation
ax4 = fig.add_subplot(gs[1,1])
nrfi_results[nrfi_results['YRFI']==0]['yrfi_probability'].hist(ax=ax4, bins=25, alpha=0.65, color=BLUE, label='Actual NRFI', density=True)
nrfi_results[nrfi_results['YRFI']==1]['yrfi_probability'].hist(ax=ax4, bins=25, alpha=0.65, color=ORANGE, label='Actual YRFI', density=True)
ax4.axvline(0.5, color='white', linestyle='--', linewidth=1.2)
ax4.set_xlabel('YRFI Probability', color=TEXT); ax4.legend(facecolor=BG, labelcolor=TEXT, fontsize=9)
sax(ax4, 'NRFI — Probability Separation')

# 5. K scatter
ax5 = fig.add_subplot(gs[1,2])
ax5.scatter(k_results['actual_k'], k_results['predicted_k'], alpha=0.25, s=14, color=PURPLE)
mx = max(k_results['actual_k'].max(), k_results['predicted_k'].max())+1
ax5.plot([0,mx],[0,mx], color=RED, linestyle='--', alpha=0.7, label='Perfect')
ax5.axhline(k_line, color=ORANGE, linestyle=':', alpha=0.8, label=f'Line: {k_line:.1f}')
ax5.set_xlabel('Actual Ks', color=TEXT); ax5.set_ylabel('Predicted Ks', color=TEXT)
ax5.legend(facecolor=BG, labelcolor=TEXT, fontsize=9)
sax(ax5, 'K Model — Predicted vs. Actual')

# 6. K rolling O/U accuracy
ax6 = fig.add_subplot(gs[2,:2])
kr = k_results.sort_values('game_date').copy()
kr['rolling_acc'] = kr['correct'].rolling(50, min_periods=15).mean()
ok = kr['correct'].mean()
ax6.fill_between(kr['game_date'], kr['rolling_acc'], alpha=0.2, color=PURPLE)
ax6.plot(kr['game_date'], kr['rolling_acc'], color=PURPLE, linewidth=2, label='50-start rolling')
ax6.axhline(ok, color=RED, linestyle='--', linewidth=1.5, label=f'Overall: {ok:.1%}')
ax6.axhline(0.50, color='gray', linestyle=':', alpha=0.6, label='50% baseline')
ax6.set_ylim(0.35,0.85); ax6.set_ylabel('O/U Accuracy', color=TEXT)
ax6.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
ax6.legend(facecolor=BG, labelcolor=TEXT, fontsize=9)
sax(ax6, f'K Total O/U — 50-Start Rolling Accuracy  (Line: {k_line:.1f} Ks)')

# 7. Error distribution
ax7 = fig.add_subplot(gs[2,2])
k_results['error'].hist(ax=ax7, bins=30, color=PURPLE, alpha=0.75, edgecolor='none')
ax7.axvline(0, color='white', linestyle='--', linewidth=1.2)
ax7.axvline(k_results['error'].mean(), color=ORANGE, linestyle='--', linewidth=1.2,
            label=f"Mean: {k_results['error'].mean():+.2f}")
ax7.set_xlabel('Predicted − Actual Ks', color=TEXT)
ax7.legend(facecolor=BG, labelcolor=TEXT, fontsize=9)
sax(ax7, 'K Model — Error Distribution')

plt.savefig('mlb_model_results_2024.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print('✅  Chart rendered and saved.')

## 8 · Monthly Accuracy Breakdown

In [ ]:
MONTH_ORDER = ['Mar','Apr','May','Jun','Jul','Aug','Sep','Oct']

def month_table(results, label):
    r = results.copy()
    r['month'] = pd.to_datetime(r['game_date']).dt.strftime('%b')
    tbl = r.groupby('month')['correct'].agg(accuracy='mean', n='count').reset_index()
    tbl['month'] = pd.Categorical(tbl['month'], categories=MONTH_ORDER, ordered=True)
    tbl = tbl.sort_values('month').dropna(subset=['month'])
    print(f'\n{label}')
    print(f'  {"Month":<8}{"Accuracy":>10}{"Games":>8}')
    print(f'  {"-"*28}')
    for _, r in tbl.iterrows():
        bar = '█' * int(r['accuracy']*100/5)
        print(f'  {str(r["month"]):<8}{r["accuracy"]*100:>8.1f}%{int(r["n"]):>7}   {bar}')

month_table(nrfi_results, 'NRFI/YRFI Model — Accuracy by Month')
month_table(k_results,    'K Total O/U Model — Accuracy by Month')

## 9 · Save Results to CSV

In [ ]:
nrfi_results.to_csv('nrfi_backtest_2024.csv', index=False)
k_results.to_csv('k_backtest_2024.csv', index=False)
print('💾  Saved: nrfi_backtest_2024.csv')
print('💾  Saved: k_backtest_2024.csv')

# Download from Colab if running there
try:
    from google.colab import files
    files.download('nrfi_backtest_2024.csv')
    files.download('k_backtest_2024.csv')
    files.download('mlb_model_results_2024.png')
    print('⬇️  Downloads triggered in Colab.')
except ImportError:
    print('(Not in Colab — files saved locally)')